In [ ]:
import os
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

# Embeddings model
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

# Vector store
vector_store = Chroma(
    collection_name="neuro_collection",
    embedding_function=embeddings,
    persist_directory="./chroma_langchain_db",  
)


## Prompts

In [ ]:
from langchain.prompts import PromptTemplate

SYSTEM_RULES = {
    "rules": [
        "Neurotechnology implants have become popular by 2075.",
        "Memory manipulation is technically possible but ethically controversial.",
        "Each year, one major neurotech-related event happens.",
        "Players always receive exactly 3 choices."
    ],
    "constraints": [
        "Scenarios must stay realistic for neurotechnology research.",
    ]
}


BASE_PROMPT = """
Follow these fixed system rules (do not change them):
{system_rules}

Context:
{context}
"""


scenario_writer_prompt = PromptTemplate.from_template(
    BASE_PROMPT + """
    You are the Scenario Writer.
    Write a ~80 word scenario describing a neurotechnology-related event in the year {year}.
    Use ONLY the provided Context above. Do NOT invent sources, names, or events not present in Context.
    The "citations" field is pre-filled with the actual filenames from the retrieved context; do not change it.

    Output JSON:
    {{
        "year": {year},
        "scenario_text": "...",
        "citations": [{citations}]
    }}
    """
)

choice_maker_prompt = PromptTemplate.from_template(
    BASE_PROMPT + """
    You are the Choice Maker.
    Base your decisions ONLY on the SCENARIO below and the Context.
    
    SCENARIO (JSON):
    {scenario}

    Output JSON:
    {{
        "choices": [
            {{"id": 1, "text": "..."}},
            {{"id": 2, "text": "..."}},
            {{"id": 3, "text": "..."}}
        ]
    }}
    """
)

outcome_prompt = PromptTemplate.from_template(
    BASE_PROMPT + """
    You are the Outcome Generator.
    Given the scenario, the available choices, and the player's choice (choice_id={choice_id}),
    describe the outcome.

    Scenario:
    {scenario}

    Choices:
    {choices}

    Output JSON:
    {{
        "choice_id": {choice_id},
        "outcome_text": "..."
    }}
    """
)


In [ ]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langgraph.graph import StateGraph
from typing_extensions import List, TypedDict
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader, Docx2txtLoader
from langgraph.graph import START
from langchain_core.messages import SystemMessage, HumanMessage
import json

DATA_DIR = Path("../neurotech")

docs = []
for path in DATA_DIR.rglob("*"):
    suf = path.suffix.lower()
    if suf == ".pdf":
        loaded = PyPDFLoader(str(path)).load()
    elif suf == ".docx":
        loaded = Docx2txtLoader(str(path)).load()
    else:
        continue
    for d in loaded:
        d.metadata = d.metadata or {}
        d.metadata["source"] = path.name
    docs.extend(loaded)


text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
all_splits = text_splitter.split_documents(docs)

all_splits = [c for c in all_splits if c.page_content and c.page_content.strip()]

_ = vector_store.add_documents(all_splits)


class State(TypedDict):
    question: str
    context: List[Document]
    answer: str
    role: str
    scenario: str
    choices: List[dict]
    choice_id: int
    year: int
    llm: object


def retrieve(state: State):
    retrieved_docs = vector_store.similarity_search(state["question"], k=4)
    return {"context": retrieved_docs}


def _as_messages(system_rules_str: str, user_prompt: str):
    return [
        SystemMessage(content=f"Follow these fixed system rules (do not change them):\n{system_rules_str}"),
        HumanMessage(content=user_prompt),
    ]


def generate(state: State):
    docs_content = "\n\n".join(doc.page_content for doc in state["context"])
    role = state["role"]
    year = state.get("year", 2075)
    llm = state["llm"]

    system_rules_str = str(SYSTEM_RULES)

    if role == "scenario":
        sources = []
        for doc in state["context"]:
            src = (doc.metadata or {}).get("source")
            if src and src not in sources:
                sources.append(src)
        if not sources:
            sources = ["(no_source_found)"]
        citations_literal = ", ".join([f'"{s}"' for s in sources])

        user_prompt = scenario_writer_prompt.format(
            context=docs_content,
            year=year,
            system_rules=SYSTEM_RULES,
            citations=citations_literal,
        )
        messages = _as_messages(system_rules_str, user_prompt)
        response = llm.invoke(messages)
        try:
            data = json.loads(response.content)
        except Exception:
            data = {}
        return {"answer": response.content, "scenario": data}

    elif role == "choices":
        user_prompt = choice_maker_prompt.format(
            context=docs_content,
            year=year,
            system_rules=SYSTEM_RULES,
            scenario=json.dumps(state.get("scenario", ""), indent=2),
        )
        messages = _as_messages(system_rules_str, user_prompt)
        response = llm.invoke(messages)
        try:
            data = json.loads(response.content)
        except Exception:
            data = {"choices": []}
        return {"answer": response.content, "choices": data.get("choices", [])}

    elif role == "outcome":
        user_prompt = outcome_prompt.format(
            context=docs_content,
            system_rules=SYSTEM_RULES,
            scenario=json.dumps(state.get("scenario", ""), indent=2),
            choices=json.dumps(state.get("choices", ""), indent=2),
            choice_id=state.get("choice_id", None),
        )
        messages = _as_messages(system_rules_str, user_prompt)
        response = llm.invoke(messages)
        try:
            data = json.loads(response.content)
        except Exception:
            data = {}
        return {"answer": response.content, "outcome": data}

    else:
        raise ValueError(f"Unknown role: {role}")


graph_builder = StateGraph(State).add_sequence([retrieve, generate])
graph_builder.add_edge(START, "retrieve")
graph = graph_builder.compile()

In [ ]:
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace

HF = os.getenv("HF_TOKEN")

common_llm = dict(
    task="conversational",
    huggingfacehub_api_token=HF,
    temperature=0.2,
    max_new_tokens=256,
)

# Base endpoints
_llama = HuggingFaceEndpoint(repo_id="meta-llama/Llama-3.1-8B-Instruct", **common_llm)
_qwen  = HuggingFaceEndpoint(repo_id="Qwen/Qwen2.5-7B-Instruct", **common_llm)
_gemma = HuggingFaceEndpoint(repo_id="google/gemma-2-9b-it", **common_llm) 


model_A = ChatHuggingFace(llm=_llama)
model_B = ChatHuggingFace(llm=_qwen)
model_C = ChatHuggingFace(llm=_gemma) 

models = {
    "llama8b": model_A,
    "qwen7b": model_B,
    "gemma9b": model_C,
}

def run_once(m):
    try:
        s = graph.invoke({"question": "Make a scenario", "role": "scenario", "year": 2076, "llm": m})
        c = graph.invoke({"question": "What are the possible choices?", "role": "choices", "scenario": s["scenario"], "llm": m})
        o = graph.invoke({"question": "What happens next?", "role": "outcome", "scenario": s["scenario"], "choices": c["choices"], "choice_id": 2, "llm": m})
        return s, c, o
    except Exception as e:
        import traceback
        print("Error with model:", e)
        traceback.print_exc()
        return None, None, None


for name, m in models.items():
    s, c, o = run_once(m)
    print(f"\n=== {name} ===")
    if s and c and o:
        print("Scenario:", s["answer"])
        print("Choices:", c["answer"])
        print("Outcome:", o["answer"])
    else:
        print("This model failed — see traceback above.")

